# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

tf.random.set_seed(42)
np.random.seed(42)

print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

device = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"
print("Using device:", device)

print_every = 700


2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Using device: /GPU:0


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы.

In [2]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

FAST_SPLIT = dict(num_training=12000, num_validation=1000, num_test=2000)
BASE_REQ_SPLIT = dict(num_training=49000, num_validation=1000, num_test=2000)
EXP_REQ_SPLIT = dict(num_training=30000, num_validation=1000, num_test=2000)
ACTIVE_SPLIT_LABEL = 'fast'

NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10(**FAST_SPLIT)
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)
print('Current split: FAST', FAST_SPLIT)


170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step
Train data shape:  (12000, 32, 32, 3)
Train labels shape:  (12000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (2000, 32, 32, 3)
Test labels shape:  (2000,)
Current split: FAST {'num_training': 12000, 'num_validation': 1000, 'num_test': 2000}


### Примечание по режимам запуска

Ноутбук использует смешанный режим, чтобы запуск оставался заметно быстрее полной версии, но обязательные пороги из задания всё равно можно было пройти.

- **Fast split**: `train=12000`, `val=1000`, `test=2000` — используется по умолчанию для большинства промежуточных проверок.
- **Base requirement split**: `train=49000`, `val=1000`, `test=2000` — включается перед базовой трехслойной CNN, где требуется `> 50%` accuracy после 1 эпохи.
- **Experiment requirement split**: `train=30000`, `val=1000`, `test=2000` — включается перед экспериментальной частью, чтобы быстрее получить `>= 70%` на валидации за 10 эпох.

После обязательных checkpoint-ов ноутбук при необходимости возвращается на быстрый режим.


In [3]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[idxs[i:i+B]], self.y[idxs[i:i+B]]) for i in range(0, N, B))


def rebuild_datasets(batch_size=64):
    global train_dset, val_dset, test_dset
    train_dset = Dataset(X_train, y_train, batch_size=batch_size, shuffle=True)
    val_dset = Dataset(X_val, y_val, batch_size=batch_size, shuffle=False)
    test_dset = Dataset(X_test, y_test, batch_size=batch_size, shuffle=False)


def set_data_split(split_cfg, label):
    global X_train, y_train, X_val, y_val, X_test, y_test, ACTIVE_SPLIT_LABEL
    X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10(**split_cfg)
    ACTIVE_SPLIT_LABEL = label
    rebuild_datasets(batch_size=64)
    print(f'Active split: {label}')
    print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)


def use_fast_split():
    set_data_split(FAST_SPLIT, 'fast')


def use_base_requirement_split():
    set_data_split(BASE_REQ_SPLIT, 'base_requirement')


def use_experiment_requirement_split():
    set_data_split(EXP_REQ_SPLIT, 'experiment_requirement')


rebuild_datasets(batch_size=64)


In [4]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети.

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [5]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()

    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации.

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU
5. Полносвязный слой
6. Функция активации Softmax

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [6]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.conv1 = tf.keras.layers.Conv2D(
            channel_1,
            kernel_size=5,
            padding="same",
            activation="relu",
            kernel_initializer=initializer
        )
        self.conv2 = tf.keras.layers.Conv2D(
            channel_2,
            kernel_size=3,
            padding="same",
            activation="relu",
            kernel_initializer=initializer
        )
        self.flatten = tf.keras.layers.Flatten()
        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation="softmax",
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################

    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(x.shape) == 4 and x.shape[1] == 3 and x.shape[-1] != 3:
            x = tf.transpose(x, [0, 2, 3, 1])
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        return scores


In [7]:
def test_ThreeLayerConvNet():
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [8]:
def reset_metric(metric):
    if hasattr(metric, "reset_state"):
        metric.reset_state()
    else:
        metric.reset_states()


def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.

    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for

    Returns: Nothing, but prints progress during trainingn
    """
    with tf.device(device):
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        t = 0
        for epoch in range(num_epochs):
            reset_metric(train_loss)
            reset_metric(train_accuracy)
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    if t % print_every == 0:
                        reset_metric(val_loss)
                        reset_metric(val_accuracy)
                        for test_x, test_y in val_dset:
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)
                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print(template.format(t, epoch + 1,
                                             train_loss.result(),
                                             train_accuracy.result() * 100,
                                             val_loss.result(),
                                             val_accuracy.result() * 100))
                    t += 1
        reset_metric(val_loss)
        reset_metric(val_accuracy)
        for test_x, test_y in val_dset:
            prediction = model(test_x, training=False)
            t_loss = loss_fn(test_y, prediction)
            val_loss.update_state(t_loss)
            val_accuracy.update_state(test_y, prediction)
        print('Final Val Loss: {}, Final Val Accuracy: {}'.format(
            val_loss.result(),
            val_accuracy.result() * 100
        ))
        globals()['last_trained_model'] = model
        globals()['last_val_accuracy'] = float(val_accuracy.result().numpy())
        return model


In [9]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

_ = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.2130789756774902, Accuracy: 10.9375, Val Loss: 2.7241053581237793, Val Accuracy: 16.0
Final Val Loss: 1.8918430805206299, Final Val Accuracy: 35.400001525878906


### Переключение на расширенную выборку для обязательного порога

Для этой проверки используется почти полная обучающая выборка, чтобы трехслойная CNN уверенно прошла требование `> 50%` accuracy после 1 эпохи.


In [10]:
use_base_requirement_split()


Active split: base_requirement
Train: (49000, 32, 32, 3) Val: (1000, 32, 32, 3) Test: (2000, 32, 32, 3)


Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 .

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [11]:
learning_rate = 1e-2
channel_1, channel_2, num_classes = 96, 64, 10

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

_ = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 2.7671127319335938, Accuracy: 14.0625, Val Loss: 30.82884407043457, Val Accuracy: 16.899999618530273
Iteration 700, Epoch 1, Loss: 1.6266711950302124, Accuracy: 44.67279052734375, Val Loss: 1.2743269205093384, Val Accuracy: 57.30000305175781
Final Val Loss: 1.296069860458374, Final Val Accuracy: 54.20000076293945


In [12]:
use_fast_split()


Active split: fast
Train: (12000, 32, 32, 3) Val: (1000, 32, 32, 3) Test: (2000, 32, 32, 3)


Возвращаемся на быструю выборку для оставшихся промежуточных пунктов, где жесткие пороги из задания не проверяются.


# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [13]:
learning_rate = 1e-2

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax',
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

_ = train_part34(model_init_fn, optimizer_init_fn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.049384593963623, Accuracy: 6.25, Val Loss: 2.7568674087524414, Val Accuracy: 16.899999618530273
Final Val Loss: 2.032456636428833, Final Val Accuracy: 36.19999694824219


Альтернативный менее гибкий способ обучения:

In [14]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - loss: 2.0864 - sparse_categorical_accuracy: 0.3207 - val_loss: 2.0457 - val_sparse_categorical_accuracy: 0.3450
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 2.0084 - sparse_categorical_accuracy: 0.3310


[2.0083532333374023, 0.3310000002384186]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [15]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
        tf.keras.layers.Conv2D(64, kernel_size=5, padding="same", activation="relu", kernel_initializer=initializer),
        tf.keras.layers.Conv2D(32, kernel_size=3, padding="same", activation="relu", kernel_initializer=initializer),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(10, activation="softmax", kernel_initializer=initializer)
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 1e-2
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

_ = train_part34(model_init_fn, optimizer_init_fn)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Iteration 0, Epoch 1, Loss: 2.8661060333251953, Accuracy: 7.8125, Val Loss: 28.941974639892578, Val Accuracy: 11.5
Final Val Loss: 1.8069905042648315, Final Val Accuracy: 35.5


In [16]:
model = model_init_fn()
model.compile(optimizer=optimizer_init_fn(),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)


188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step - loss: 1.9836 - sparse_categorical_accuracy: 0.3202 - val_loss: 1.7130 - val_sparse_categorical_accuracy: 0.3710
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 1.6989 - sparse_categorical_accuracy: 0.3950


[1.6989142894744873, 0.39500001072883606]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры.

Ниже представлен пример для полносвязной сети.

In [17]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)

    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)

    with tf.device(device):
        scores = model(x)
        print(scores.shape)

test_two_layer_fc_functional()

(64, 10)


In [18]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

_ = train_part34(model_init_fn, optimizer_init_fn)


Iteration 0, Epoch 1, Loss: 3.245587110519409, Accuracy: 6.25, Val Loss: 2.662126064300537, Val Accuracy: 16.799999237060547
Final Val Loss: 2.2799293994903564, Final Val Accuracy: 33.599998474121094


Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут).

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

### Переключение на расширенную выборку для экспериментальной части

Для двух финальных экспериментов используется увеличенная обучающая выборка. Это заметно быстрее полной версии, но дает существенно более устойчивое качество, чем fast split.


In [19]:
use_experiment_requirement_split()


Active split: experiment_requirement
Train: (30000, 32, 32, 3) Val: (1000, 32, 32, 3) Test: (2000, 32, 32, 3)


### Эксперимент 1

Сначала проверяется более простая сверточная сеть без Batch Normalization и Dropout, чтобы сравнить ее с более сильной архитектурой из следующего эксперимента.

In [20]:
class CustomConvNetExperiment1(tf.keras.Model):
    def __init__(self):
        super(CustomConvNetExperiment1, self).__init__()
        self.conv1 = tf.keras.layers.Conv2D(16, (3, 3), padding='same', activation='relu')
        self.pool1 = tf.keras.layers.MaxPooling2D((2, 2))
        self.conv2 = tf.keras.layers.Conv2D(32, (3, 3), padding='same', activation='relu')
        self.pool2 = tf.keras.layers.MaxPooling2D((2, 2))
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(128, activation='relu')
        self.fc2 = tf.keras.layers.Dense(10, activation='softmax')

    def call(self, input_tensor, training=False):
        x = self.conv1(input_tensor)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


learning_rate = 1e-3
print_every = 100

def model_init_fn():
    return CustomConvNetExperiment1()

def optimizer_init_fn():
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)

experiment1_model = train_part34(model_init_fn, optimizer_init_fn, num_epochs=10, is_training=True)
experiment1_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.sparse_categorical_accuracy]
)
experiment1_model.evaluate(X_test, y_test, batch_size=64)

Iteration 0, Epoch 1, Loss: 2.359668731689453, Accuracy: 17.1875, Val Loss: 2.375117301940918, Val Accuracy: 12.5
Iteration 100, Epoch 1, Loss: 1.7969987392425537, Accuracy: 35.39603805541992, Val Loss: 1.6163285970687866, Val Accuracy: 40.599998474121094
Iteration 200, Epoch 1, Loss: 1.6372497081756592, Accuracy: 41.02922821044922, Val Loss: 1.440301537513733, Val Accuracy: 49.29999923706055
Iteration 300, Epoch 1, Loss: 1.5454840660095215, Accuracy: 44.33139419555664, Val Loss: 1.3341546058654785, Val Accuracy: 52.20000076293945
Iteration 400, Epoch 1, Loss: 1.4800996780395508, Accuracy: 46.956825256347656, Val Loss: 1.2613325119018555, Val Accuracy: 54.900001525878906
Iteration 500, Epoch 2, Loss: 1.1046688556671143, Accuracy: 61.279296875, Val Loss: 1.2149747610092163, Val Accuracy: 57.79999923706055
Iteration 600, Epoch 2, Loss: 1.1418391466140747, Accuracy: 59.69459915161133, Val Loss: 1.1407842636108398, Val Accuracy: 60.20000076293945
Iteration 700, Epoch 2, Loss: 1.12701869010

[1.1799465417861938, 0.640500009059906]

### Эксперимент 2

Во втором эксперименте используется более глубокая архитектура с Batch Normalization, MaxPooling, Dropout и оптимизатором Adam.

In [21]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.keras.initializers.HeNormal()
        self.conv1 = tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()
        self.conv2 = tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D()
        self.drop1 = tf.keras.layers.Dropout(0.25)
        self.conv3 = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()
        self.conv4 = tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D()
        self.drop2 = tf.keras.layers.Dropout(0.30)
        self.conv5 = tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn5 = tf.keras.layers.BatchNormalization()
        self.conv6 = tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False, kernel_initializer=initializer)
        self.bn6 = tf.keras.layers.BatchNormalization()
        self.pool3 = tf.keras.layers.MaxPooling2D()
        self.drop3 = tf.keras.layers.Dropout(0.40)
        self.flatten = tf.keras.layers.Flatten()
        self.fc1 = tf.keras.layers.Dense(256, use_bias=False, kernel_initializer=initializer)
        self.bn7 = tf.keras.layers.BatchNormalization()
        self.drop4 = tf.keras.layers.Dropout(0.50)
        self.fc2 = tf.keras.layers.Dense(10, activation="softmax")

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################

    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)
        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)
        x = self.conv5(x)
        x = self.bn5(x, training=training)
        x = tf.nn.relu(x)
        x = self.conv6(x)
        x = self.bn6(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool3(x)
        x = self.drop3(x, training=training)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn7(x, training=training)
        x = tf.nn.relu(x)
        x = self.drop4(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate=learning_rate)

custom_model = train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)
custom_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.sparse_categorical_accuracy]
)
custom_model.evaluate(X_test, y_test, batch_size=64)


Iteration 0, Epoch 1, Loss: 3.186471939086914, Accuracy: 10.9375, Val Loss: 5.486163139343262, Val Accuracy: 10.40000057220459
Iteration 700, Epoch 2, Loss: 1.3104617595672607, Accuracy: 53.28663635253906, Val Loss: 1.2019211053848267, Val Accuracy: 57.900001525878906
Iteration 1400, Epoch 3, Loss: 1.055966854095459, Accuracy: 62.52699661254883, Val Loss: 0.9983920454978943, Val Accuracy: 66.5
Iteration 2100, Epoch 5, Loss: 0.8694659471511841, Accuracy: 69.09722137451172, Val Loss: 0.8348257541656494, Val Accuracy: 70.70000457763672
Iteration 2800, Epoch 6, Loss: 0.7881739139556885, Accuracy: 72.2073745727539, Val Loss: 0.760244607925415, Val Accuracy: 72.89999389648438
Iteration 3500, Epoch 8, Loss: 0.6956683993339539, Accuracy: 75.09317016601562, Val Loss: 0.6639474034309387, Val Accuracy: 77.20000457763672
Iteration 4200, Epoch 9, Loss: 0.6621986627578735, Accuracy: 76.93833923339844, Val Loss: 0.6575797200202942, Val Accuracy: 77.30000305175781
Final Val Loss: 0.6300690174102783, F

[0.611699104309082, 0.7829999923706055]

## Вывод

В лабораторной работе были реализованы и проверены несколько моделей классификации изображений CIFAR-10 с использованием TensorFlow/Keras: через Model Subclassing API, Sequential API и Functional API. Для ускорения выполнения использовался смешанный режим запуска: промежуточные части обучались на уменьшенной выборке, а этапы с обязательными требованиями по качеству — на расширенных выборках.

Полносвязные модели показали сравнительно невысокое качество: TwoLayerFC через Model Subclassing API достигла 35.4% accuracy на валидационной выборке, через Sequential API — 36.2%, а модель через Functional API — 33.6%. Это ожидаемо, так как полносвязные сети не учитывают пространственную структуру изображения.

Трехслойная сверточная сеть, реализованная через Model Subclassing API и обученная с помощью SGD с Nesterov momentum = 0.9, достигла 54.2% accuracy на валидационной выборке после одной эпохи обучения, тем самым выполнив обязательное требование задания (> 50%). Аналогичная модель через Sequential API показала более низкий результат на уменьшенной выборке, что объясняется использованием fast split для промежуточных этапов.

В экспериментальной части были рассмотрены две сверточные архитектуры. Первая, более простая модель без Batch Normalization и Dropout, достигла 67.0% accuracy на валидационной выборке и 64.05% на тестовой выборке. Вторая, более глубокая модель с Batch Normalization, MaxPooling, Dropout и оптимизатором Adam, показала лучший результат: 77.3% accuracy на валидационной выборке и 78.3% на тестовой выборке. Таким образом, требование получить не менее 70% accuracy на validation за 10 эпох было выполнено.

По итогам работы можно сделать вывод, что сверточные нейронные сети значительно лучше подходят для классификации изображений CIFAR-10, чем простые полносвязные модели. Усложнение архитектуры, добавление Batch Normalization и Dropout, а также использование оптимизатора Adam позволяют заметно повысить качество классификации и сделать обучение более устойчивым.